In [1]:
import os
import pandas as pd
import kagglehub

print("🔄 Step 1: Downloading dataset programmatically via kagglehub...")
# Downloads the dataset to your local cache automatically
dataset_path = kagglehub.dataset_download("grikomsn/amazon-cell-phones-reviews")
print(f"✅ Success! Data cached at: {dataset_path}\n")

# Connect the paths to the file names
items_path = os.path.join(dataset_path, "20191226-items.csv")
reviews_path = os.path.join(dataset_path, "20191226-reviews.csv")

print("🔄 Step 2: Loading CSV datasets into Pandas DataFrames...")
df_items = pd.read_csv(items_path)
df_reviews = pd.read_csv(reviews_path)
print(f"📊 Items Shape: {df_items.shape} | Reviews Shape: {df_reviews.shape}\n")

print("🔄 Step 3: Merging data on unique column ('asin')...")
# Merge the review texts with the brand/product metadata
df_master = pd.merge(df_reviews, df_items, on='asin', how='inner')

# Filter down to the exact columns needed for AI Modeling & Power BI dashboards
df_filtered = df_master[['body', 'rating_x', 'brand']].dropna()
df_filtered.columns = ['review_text', 'rating', 'brand']

print(f"✅ Master pipeline complete!")
print(f"📈 Total combined rows ready for NLP: {df_filtered.shape[0]}")
print("\n--- Previewing your dataframe below ---")
df_filtered.head(5)

C:\Users\Pankaj\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Pankaj\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


🔄 Step 1: Downloading dataset programmatically via kagglehub...
✅ Success! Data cached at: C:\Users\Pankaj\.cache\kagglehub\datasets\grikomsn\amazon-cell-phones-reviews\versions\2

🔄 Step 2: Loading CSV datasets into Pandas DataFrames...
📊 Items Shape: (720, 10) | Reviews Shape: (67986, 8)

🔄 Step 3: Merging data on unique column ('asin')...
✅ Master pipeline complete!
📈 Total combined rows ready for NLP: 67760

--- Previewing your dataframe below ---


,review_text,rating,brand
14,DON'T BUY OUT OF SERVICE,1,Motorola
15,I have been with nextel for nearly a year now ...,4,Motorola
16,"I just got it and have to say its easy to use,...",5,Motorola
17,1 star because the phones locked so I have to ...,1,Motorola
18,The product has been very good. I had used thi...,5,Motorola


### NLP Text processing

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK resource packages 
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Initialize NLP processing tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Force lowercase conversion
    text = str(text).lower()
    # 2. Strip out HTML elements, URL links, and special symbols
    text = re.sub(r'<.*?>|https?://\s+|www\.\s+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Tokenize words, filter out stop words, and apply lemmatization
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return ' '.join(cleaned_words)

print("🔄 Processing 67k+ text reviews... (This may take 15-30 seconds)")
# Apply our pipeline structure across the dataframe
df_filtered['cleaned_review'] = df_filtered['review_text'].apply(clean_text)
print("✅ Text preprocessing completed successfully!")

# Show a direct comparison layout of raw vs cleaned text
print("\n--- Verifying Text Transformation (Before vs After) ---")
print(df_filtered[['review_text', 'cleaned_review']].head(3))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pankaj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Pankaj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Pankaj\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


🔄 Processing 67k+ text reviews... (This may take 15-30 seconds)
✅ Text preprocessing completed successfully!

--- Verifying Text Transformation (Before vs After) ---
                                          review_text  \
14                           DON'T BUY OUT OF SERVICE   
15  I have been with nextel for nearly a year now ...   
16  I just got it and have to say its easy to use,...   

                                       cleaned_review  
14                                   dont buy service  
15  nextel nearly year started time last year moto...  
16  got say easy use hear person talking fine prob...  


### Building ml model

#### TF-IDF Vectorization

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pickle

print("🚩 Step 1: Creating 3-class target labels...")
# Industry Standard Mapping: 4-5 -> 2 (Positive), 3 -> 1 (Neutral), 1-2 -> 0 (Negative)
def assign_label(rating):
    if rating >= 4:
        return 2
    elif rating == 3:
        return 1
    else:
        return 0

df_filtered['target'] = df_filtered['rating'].apply(assign_label)

print("🚩 Step 2: Splitting dataset into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['cleaned_review'],
    df_filtered['target'],
    test_size=0.2,
    random_state=42,
    stratify=df_filtered['target'] # Ensures balanced data distribution across all three classes
)
print("🚩 Step 3: Transforming text data into numerical features (TF-IDF)...")
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("🚩 Step 4: Training Logistic Regression Model...")
# Logistic Regression automatically optimizes for multi-class classification
ml_model = LogisticRegression(max_iter=1000, random_state=42)
ml_model.fit(X_train_tfidf, y_train)

print("🚩 Step 5: Generating model evaluation metrics...")
y_pred = ml_model.predict(X_test_tfidf)
print(f"📊 Accuracy Score: {accuracy_score(y_test, y_pred):.2f}")

print("\n--- Detailed Classification Report ---")
# Added 'Neutral' to align with the 3-class system
print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

print("🚩 Step 6: Overwriting model assets locally...")
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('sentiment_ml_model.pkl', 'wb') as f:
    pickle.dump(ml_model, f)
print("✅ Success: Clean, high-accuracy 3-class model assets saved successfully!")

🚩 Step 1: Creating 3-class target labels...
🚩 Step 2: Splitting dataset into train and test sets...
🚩 Step 3: Transforming text data into numerical features (TF-IDF)...
🚩 Step 4: Training Logistic Regression Model...
🚩 Step 5: Generating model evaluation metrics...
📊 Accuracy Score: 0.86

--- Detailed Classification Report ---
              precision    recall  f1-score   support

    Negative       0.79      0.81      0.80      3320
     Neutral       0.46      0.06      0.10       948
    Positive       0.89      0.96      0.92      9284

    accuracy                           0.86     13552
   macro avg       0.71      0.61      0.61     13552
weighted avg       0.83      0.86      0.83     13552

🚩 Step 6: Overwriting model assets locally...
✅ Success: Clean, high-accuracy 3-class model assets saved successfully!


### Testing

In [7]:
import pickle

# 1. Load the saved model and vectorizer from the project directory
with open('tfidf_vectorizer.pkl', 'rb') as f:
    loaded_vectorizer = pickle.load(f)
with open('sentiment_ml_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# 2. Define a function to process and test any sentence instantly
def predict_custom_review(sample_review):
    # Industry Standard Rule: Always apply the same text preprocessing before prediction
    cleaned_sample = clean_text(sample_review)
    review_vectorized = loaded_vectorizer.transform([cleaned_sample])
    
    # Generate prediction value
    prediction = loaded_model.predict(review_vectorized)[0]
    
    # 3-Class Sentiment Mapping Breakdown
    if prediction == 2:
        sentiment = "😊 Positive (Happy Customer)"
    elif prediction == 1:
        sentiment = "😐 Neutral (Satisfied/Factual)"
    else:
        sentiment = "😭 Negative (Unhappy Customer)"
        
    print(f"📝 Original Review Text: \"{sample_review}\"")
    print(f"🔮 Predicted Sentiment  : {sentiment}\n")

# 3. Test it with three distinct industry benchmark inputs
predict_custom_review("The camera quality is absolutely stunning and delivery was fast!")
predict_custom_review("The package contains a data cable, a charging brick, and a user manual.")
predict_custom_review("Worst purchase ever. The screen was cracked and it keeps rebooting.")

📝 Original Review Text: "The camera quality is absolutely stunning and delivery was fast!"
🔮 Predicted Sentiment  : 😊 Positive (Happy Customer)

📝 Original Review Text: "The package contains a data cable, a charging brick, and a user manual."
🔮 Predicted Sentiment  : 😊 Positive (Happy Customer)

📝 Original Review Text: "Worst purchase ever. The screen was cracked and it keeps rebooting."
🔮 Predicted Sentiment  : 😭 Negative (Unhappy Customer)



In [8]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("reviews_database.db")
df = pd.read_sql_query("SELECT * FROM customer_reviews", conn)
df.to_excel("reviews_data.xlsx", index=False)
conn.close()
print("✅ Excel file created successfully!")

✅ Excel file created successfully!


In [9]:
import shutil

# Copy the file from the hidden cache folder directly into your current project directory
shutil.copy(reviews_path, "reviews_backup.csv")
print("✅ Success! 'reviews_backup.csv' has been copied to your main project folder!")

✅ Success! 'reviews_backup.csv' has been copied to your main project folder!
